<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB03_NumPy_Array_Mechanics_Dtypes_Indexing_and_Views_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB03 · Clase 3 — Mecánica de arrays en NumPy: dtypes, indexado y vistas**

## Bloque 2: IA — Machine Learning (continuación)

`NB02` usó arrays de NumPy para creación básica, indexado y operaciones matemáticas vectorizadas — suficiente para sacar una primera gráfica en pantalla, pero no suficiente para usar NumPy con confianza real. Todos los modelos que entrena este curso, desde la primera regresión de `NB07` hasta la red neuronal más profunda, están construidos sobre arrays por debajo. Esta clase profundiza un nivel más en el propio array: qué controla realmente un `dtype`, la diferencia entre una **vista** y una **copia** (una fuente real y frecuente de bugs silenciosos), y los patrones de indexado fancy/booleano que se usan constantemente con datos reales.

Esta clase trabaja enteramente con datos ya usados en otras partes del curso — `sonar.all-data` (60 características acústicas reales, de `NB08`) y `ship_fuel_efficiency.csv` (de `NB07`) — así que cada ejemplo está anclado en un dataset real, no en números de juguete.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Explicar qué es un `dtype` y por qué importa para la memoria y la corrección.
- Crear arrays con la forma y el tipo adecuados para una tarea dada (`zeros`, `ones`, `arange`, `linspace`, `meshgrid`, `identity`).
- Indexar y hacer slicing de arrays 1-D y multidimensionales con confianza, incluyendo índices negativos y pasos.
- Explicar, y demostrar con código real, la diferencia entre una **vista** y una **copia** de NumPy — y por qué importa.
- Usar máscaras booleanas e indexado fancy para seleccionar datos reales por condición, no solo por posición.

### Agenda (clase de 2 horas)

| # | Sección | Minutos |
|---|---|---|
| 1 | Repaso y por qué importa la mecánica de los arrays | 10 |
| 2 | dtypes: qué son y por qué importan | 15 |
| 3 | Patrones de creación de arrays | 15 |
| 4 | Indexado y slicing, 1-D y multi-D | 15 |
| 5 | Vistas frente a copias (un bug real y frecuente) | 15 |
| 6 | Máscaras booleanas e indexado fancy sobre datos reales | 20 |
| 7 | Reshape de arrays y reducciones por eje | 15 |
| 8 | Resumen, tarea, próxima clase | 15 |

Como siempre: orientación aproximada, no un guion cerrado.

---

## 1. Por qué importa la mecánica de los arrays

Todo dataset de este curso termina convertido en un array de NumPy (directamente, o dentro de un DataFrame de Pandas, que envuelve arrays de NumPy columna por columna). Un modelo que silenciosamente recibe el trozo de datos equivocado, o que comparte memoria por accidente sin querer, produce resultados incorrectos **sin ningún error** — `posiblemente peor que un fallo (crash), porque nada te avisa de que ha pasado`. Esta clase trata sobre la mecánica que evita esa clase de bug.

> **Para saber más**: [Conceptos básicos de arrays de NumPy (Wikipedia)](https://en.wikipedia.org/wiki/NumPy) | [documentación oficial de NumPy](https://numpy.org/doc/stable/)

Aquí va una primera muestra, pequeña, de exactamente ese tipo de bug silencioso — sin mensaje de error, solo un número que queda mal en silencio:

In [ ]:
import numpy as np

sensor_log = np.array([12.1, 12.3, 12.0, 45.8, 12.4, 11.9])
last_five = sensor_log[1:]          # looks like an independent "the rest of the log"...

sensor_log[1] = 999.0               # ...a later fix to the very first reading...
print("'last_five', after fixing something else entirely:", last_five)  # ...changed anyway, silently


Sin error, sin aviso — `last_five` cambió porque el slicing comparte memoria. La Sección 5 explica exactamente por qué, y cómo evitarlo.

---

## 2. dtypes: qué son y por qué importan

Todo array de NumPy tiene un único `dtype` (tipo de dato) compartido por todos sus elementos — a diferencia de una lista de Python, que puede mezclar tipos libremente. Esto es exactamente lo que hace rápido a NumPy: saber que cada elemento es, por ejemplo, un float de 64 bits permite a NumPy operar sobre todo el array en bucles vectorizados y compactos, en vez de comprobar el tipo de cada elemento uno a uno.

La siguiente celda carga las 60 características acústicas reales del dataset Sonar e inspecciona su dtype directamente.

In [ ]:
import numpy as np
import urllib.request

sonar_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/sonar.all-data"
raw_lines = urllib.request.urlopen(sonar_url).read().decode("utf-8").strip().split("\n")

sonar_features = np.array([[float(x) for x in line.split(",")[:-1]] for line in raw_lines])
sonar_labels = np.array([line.split(",")[-1] for line in raw_lines])

print(f"Shape: {sonar_features.shape}  (rows=readings, cols=frequency bands)")
print(f"dtype: {sonar_features.dtype}")
print(f"Memory used: {sonar_features.nbytes / 1024:.1f} KB")


La siguiente celda demuestra el coste real en memoria de la elección de dtype: el mismo array 208x60 de valores, almacenado como floats de 64 bits (el valor por defecto de NumPy) frente a floats de 32 bits. Reducir la precisión a la mitad reduce realmente la memoria a la mitad — `un compromiso real y práctico en cuanto un dataset es lo bastante grande como para que importe` (irrelevante aquí, con ~100 KB, pero muy real para las rejillas de varios GB que descargan `NB18`/`NB20`).

In [ ]:
sonar_f32 = sonar_features.astype(np.float32)

print(f"float64: {sonar_features.nbytes / 1024:.1f} KB")
print(f"float32: {sonar_f32.nbytes / 1024:.1f} KB")
print(f"Values still close after the precision drop? {np.allclose(sonar_features, sonar_f32, atol=1e-6)}")


Otra trampa habitual de los dtypes, que merece la pena ver una vez directamente: **la división entre enteros trunca, y los arrays de enteros pueden desbordar su rango en silencio** de formas que los arrays de float no. Ninguna de las dos cosas lanza un error.

In [ ]:
int_array = np.array([1, 2, 3], dtype=np.int32)
float_array = np.array([1, 2, 3], dtype=np.float64)

print("Integer array / 2:", int_array / 2)      # NumPy promotes to float on true division -- safe
print("Integer array dtype after /:", (int_array / 2).dtype)

small_int8 = np.array([120], dtype=np.int8)      # int8 range: -128 to 127
print("\nint8 value:", small_int8[0])
print("int8 value + 20 (overflows silently):", (small_int8 + np.int8(20))[0])


> **Para saber más**: [documentación de dtype de NumPy](https://numpy.org/doc/stable/reference/arrays.dtypes.html) | [Desbordamiento de enteros (Wikipedia)](https://en.wikipedia.org/wiki/Integer_overflow)

---

## 3. Patrones de creación de arrays

`El código real rara vez escribe a mano cada elemento de un array.` Estas funciones de creación cubren la mayoría de las situaciones reales: marcadores de posición para rellenar más tarde (`zeros`/`ones`/`empty`), secuencias regulares (`arange`/`linspace`), rejillas de coordenadas para cualquier cosa espacial (`meshgrid` — la herramienta exacta que usan `NB18` y `NB20` para construir una rejilla de lat/lon a partir de un fichero real meteorológico o de batimetría), y matrices identidad (`identity`, necesarias para el álgebra lineal de `NB04`).

In [ ]:
zeros_arr = np.zeros((3, 4))              # placeholder, e.g. "no readings yet" for 3 sensors x 4 timestamps
ones_arr = np.ones(5)                     # e.g. initial weights, all equal
arange_arr = np.arange(0, 24, 3)          # every 3rd hour in a 24-hour day
linspace_arr = np.linspace(0, 100, 5)     # 5 evenly-spaced depth levels from 0 to 100 m
identity_arr = np.identity(3)             # 3x3 identity matrix, needed in NB04

print("zeros:\n", zeros_arr)
print("\nones:", ones_arr)
print("\narange (every 3rd hour):", arange_arr)
print("\nlinspace (5 depth levels, 0-100 m):", linspace_arr)
print("\nidentity:\n", identity_arr)


`meshgrid` merece su propio ejemplo: convierte dos arrays de coordenadas 1-D en dos rejillas 2-D, una con la coordenada x de cada punto y otra con la coordenada y de cada punto — exactamente lo que necesitaba `NB18` para convertir los arrays separados de latitud/longitud de ERA5 en una tabla plana de pares (lat, lon), y lo que necesitaba `NB20` para la rejilla de batimetría GMRT.

In [ ]:
lon_1d = np.array([-1.0, -0.5, 0.0])
lat_1d = np.array([37.0, 37.5])

lon_grid, lat_grid = np.meshgrid(lon_1d, lat_1d)

print("1-D longitude:", lon_1d)
print("1-D latitude:", lat_1d)
print("\n2-D longitude grid (one row per latitude):\n", lon_grid)
print("\n2-D latitude grid (one column per longitude):\n", lat_grid)
print("\nEvery (lon_grid[i,j], lat_grid[i,j]) pair is one real grid point, e.g. point (0,0):",
      (lon_grid[0, 0], lat_grid[0, 0]))


> **Para saber más**: [documentación de `numpy.meshgrid`](https://numpy.org/doc/stable/reference/generated/numpy.meshgrid.html) | [documentación de `numpy.linspace`](https://numpy.org/doc/stable/reference/generated/numpy.linspace.html)

---

## 4. Indexado y slicing, 1-D y multi-D

El slicing de NumPy extiende la propia sintaxis `list[start:stop:step]` de Python `a varias dimensiones a la vez, separando cada eje con una coma`. La siguiente celda lo aplica directamente al array real de Sonar (208 lecturas x 60 bandas de frecuencia).

In [ ]:
print("First reading, all 60 bands:", sonar_features[0].shape)
print("First 5 readings, first 3 bands:\n", sonar_features[:5, :3])
print("\nEvery other reading, last band:", sonar_features[::2, -1][:5], "...")
print("\nLast reading, reversed band order, first 5 values:", sonar_features[-1, ::-1][:5])


Un error real y frecuente: asumir que `array[i][j]` y `array[i, j]` se comportan siempre igual. Para un array de NumPy 2-D normal dan el mismo valor, pero `array[i][j]` lo hace en **dos pasos separados** (primero extrae la fila `i` como su propio array, y después indexa dentro de ella) mientras que `array[i, j]` lo hace en **un solo paso** — una diferencia que importa tanto para la velocidad como, como muestra la siguiente sección, para si estás viendo una vista o una copia nueva.

In [ ]:
row_then_col = sonar_features[0][5]   # two steps: slice row 0, then index into it
direct = sonar_features[0, 5]         # one step: index both axes at once

print("Same value either way:", row_then_col == direct)
print("But array[0] alone is itself a real, separate array object:")
print("  type(sonar_features[0]):", type(sonar_features[0]))
print("  sonar_features[0].shape:", sonar_features[0].shape)


**Pruébalo tú mismo**: usando las reglas de indexado de arriba, escribe una línea que seleccione cada 10ª lectura (filas) y solo las primeras 20 bandas de frecuencia (columnas) de `sonar_features`. Predice la forma resultante antes de ejecutar tu línea, y luego compruébala con la celda de abajo.

In [ ]:
every_10th_first_20 = sonar_features[::10, :20]
print("Shape:", every_10th_first_20.shape)   # predicted: 21 rows (208 readings / 10, rounded up), 20 columns


> **Para saber más**: [documentación de indexado de NumPy](https://numpy.org/doc/stable/user/basics.indexing.html)

---

## 5. Vistas frente a copias (un bug real y frecuente)

Este es, con diferencia, el bug real más frecuente de NumPy para quien empieza con él: **el slicing básico (`array[a:b]`) devuelve una vista — una ventana sobre la *misma* memoria subyacente, no un array nuevo.** Modificar una vista modifica el original. `np.array()` o `.copy()` sobre un slice crea, en cambio, una copia genuina e independiente. `Ni la vista ni la copia imprimen ningún aviso sobre cuál de las dos tienes` — la única forma de saberlo es comprobarlo, o conocer la regla.

In [ ]:
original = sonar_features[:5, :3].copy()   # protect the real data before this demo modifies anything
demo = original.copy()

view = demo[0]          # a VIEW: shares memory with `demo`
copy = demo[1].copy()   # a genuine COPY: independent memory

print("Before modification:")
print("demo[0]:", demo[0])
print("demo[1]:", demo[1])

view[0] = -999.0        # modifying the view...
copy[0] = -999.0        # ...and modifying the copy

print("\nAfter modifying `view` and `copy`:")
print("demo[0] (changed! `view` shared its memory):", demo[0])
print("demo[1] (unchanged -- `copy` was independent):", demo[1])


El diagrama de abajo hace explícito el compartir memoria: una vista es un segundo *nombre* que apunta al mismo bloque de memoria real, mientras que una copia es un bloque completamente separado.

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))

for ax, title, shares in zip(axes, ["View: demo[0]", "Copy: demo[1].copy()"], [True, False]):
    ax.add_patch(plt.Rectangle((0.05, 0.55), 0.4, 0.3, fill=True, facecolor="lightsteelblue", edgecolor="black"))
    ax.text(0.25, 0.7, "demo\n(memory block A)", ha="center", va="center", fontsize=9)
    if shares:
        ax.add_patch(plt.Rectangle((0.55, 0.55), 0.4, 0.3, fill=False, edgecolor="darkred", linewidth=2))
        ax.text(0.75, 0.7, "view\n(points into A)", ha="center", va="center", fontsize=9, color="darkred")
        ax.annotate("", xy=(0.55, 0.7), xytext=(0.45, 0.7), arrowprops=dict(arrowstyle="->", color="darkred"))
    else:
        ax.add_patch(plt.Rectangle((0.55, 0.15), 0.4, 0.3, fill=True, facecolor="mistyrose", edgecolor="darkred"))
        ax.text(0.75, 0.3, "copy\n(memory block B)", ha="center", va="center", fontsize=9, color="darkred")
        ax.annotate("", xy=(0.6, 0.4), xytext=(0.3, 0.55), arrowprops=dict(arrowstyle="->", color="gray", linestyle="--"))
        ax.text(0.45, 0.5, "data\ncopied", ha="center", fontsize=7, color="gray")
    ax.set_xlim(0, 1)
    ax.set_ylim(0, 1)
    ax.axis("off")
    ax.set_title(title, fontsize=10)

plt.suptitle("A view shares memory with the original; a copy does not")
plt.tight_layout()
plt.show()


**Regla práctica**: siempre que hagas slicing de un array y quieras modificar el resultado *sin* tocar el original, llama a `.copy()` explícitamente. Las rejillas espaciales de `NB18` y `NB20`, y cada división train/test de este curso, dependen de que esta regla se cumpla — un modelo entrenado por accidente con datos corrompidos por una modificación de vista no intencionada sería un bug muy difícil de rastrear.

> **Para saber más**: [documentación de copias y vistas de NumPy](https://numpy.org/doc/stable/user/basics.copies.html)

---

## 6. Máscaras booleanas e indexado fancy sobre datos reales

Una **máscara booleana** es un array de valores `True`/`False`, con la misma forma que los datos, usado para seleccionar elementos por *condición* en vez de por posición — la herramienta detrás de cada filtro tipo `df[df["columna"] > x]` que ha usado este curso desde `NB02`. La siguiente celda la aplica directamente a las etiquetas reales de Sonar.

In [ ]:
is_mine = sonar_labels == "M"
print(f"Real mine readings: {is_mine.sum()} / {len(sonar_labels)} ({is_mine.mean():.1%})")

mine_readings = sonar_features[is_mine]
rock_readings = sonar_features[~is_mine]

print(f"\nMean value of band 0, mines:  {mine_readings[:, 0].mean():.4f}")
print(f"Mean value of band 0, rocks:  {rock_readings[:, 0].mean():.4f}")


El **indexado fancy** va un paso más allá: en vez de una máscara booleana, se pasa un *array explícito de posiciones* a seleccionar (y se pueden repetir o reordenar). La siguiente celda lo usa para extraer solo 5 bandas de frecuencia arbitrarias de cada lectura a la vez — el mismo mecanismo que usaría el ranking de importancia de características de `NB08` para extraer las bandas *más informativas*, una vez ordenadas.

In [ ]:
some_band_indices = np.array([10, 11, 35, 44, 48])  # illustrative subset of the 60 real frequency bands

subset_only = sonar_features[:, some_band_indices]
print(f"Full data shape: {sonar_features.shape}")
print(f"5-bands-only shape: {subset_only.shape}")
print("\nFirst reading, those 5 bands only:", subset_only[0])


Una diferencia importante respecto al slicing: **el indexado fancy siempre devuelve una copia, nunca una vista** — como los elementos seleccionados no tienen por qué ser contiguos en memoria, NumPy no tiene otra opción que construir un array nuevo. `Merece la pena saberlo con precisión porque es justo lo contrario de la regla de la Sección 5 para el slicing básico`.

In [ ]:
fancy_result = sonar_features[[0, 1, 2]]
print("Fancy indexing shares memory with the original?", np.shares_memory(fancy_result, sonar_features))

slice_result = sonar_features[0:3]
print("Basic slicing shares memory with the original?", np.shares_memory(slice_result, sonar_features))


> **Para saber más**: [documentación de indexado booleano y fancy de NumPy](https://numpy.org/doc/stable/user/basics.indexing.html#advanced-indexing) | [documentación de `numpy.shares_memory`](https://numpy.org/doc/stable/reference/generated/numpy.shares_memory.html)

---

## 7. Reshape de arrays y reducciones por eje

Dos preguntas más de mecánica que surgen constantemente en cuanto los datos reales son genuinamente multidimensionales: **`.reshape()`** — cambiar la forma de un array sin cambiar sus datos — y **elegir un eje** para una reducción como `.mean()` o `.sum()`. El `fleet_specs.mean(axis=0)` de `NB02` ya usó `axis=0` sin explicarlo del todo — aquí va la explicación completa, sobre los datos reales de Sonar.

> **Para saber más**: [documentación de `numpy.reshape`](https://numpy.org/doc/stable/reference/generated/numpy.reshape.html) | [documentación de `numpy.ndarray.sum`](https://numpy.org/doc/stable/reference/generated/numpy.ndarray.sum.html)

In [ ]:
flat = sonar_features[:3].flatten()   # take first 3 readings, flatten to 1-D (3*60 = 180 values)
print("Flattened shape:", flat.shape)

back_to_2d = flat.reshape(3, 60)      # reshape back to (3 readings, 60 bands) -- same data, new shape
print("Reshaped back:", back_to_2d.shape)
print("Same values as the original 3 readings?", np.array_equal(back_to_2d, sonar_features[:3]))

print("\nreshape(-1, 1) turns any array into a single column:")
print(sonar_features[0, :5].reshape(-1, 1))


`reshape()` nunca cambia los datos, solo cómo se agrupan en filas/columnas — el número total de elementos debe mantenerse igual (`3 x 60 = 180`, coincidiendo con `flat.shape`). `-1` le dice a NumPy "calcula tú esta dimensión", útil cuando solo una dimensión está fijada.

Las reducciones como `.mean()` y `.sum()` necesitan saber *qué* dimensión colapsar. `axis=0` colapsa a lo largo de las filas (un resultado por columna); `axis=1` colapsa a lo largo de las columnas (un resultado por fila).

In [ ]:
print("Overall mean across the whole dataset:", sonar_features.mean())

mean_per_band = sonar_features.mean(axis=0)     # axis=0: collapse rows -> one mean per frequency band (60 values)
mean_per_reading = sonar_features.mean(axis=1)  # axis=1: collapse columns -> one mean per reading (208 values)

print("\nmean(axis=0) shape (one value per band):", mean_per_band.shape)
print("mean(axis=1) shape (one value per reading):", mean_per_reading.shape)

print("\nWhich frequency band has the highest average energy across all 208 readings?")
loudest_band = mean_per_band.argmax()
print(f"Band {loudest_band}, mean value {mean_per_band[loudest_band]:.4f}")


**Pruébalo tú mismo, antes de ejecutar la siguiente celda**: usando `mean_per_reading` de arriba, ¿qué única *lectura* (fila) tiene el valor medio más alto en las 60 bandas — una mina o una roca? Escribe tú mismo esa línea de código, y luego compárala con la celda de abajo.

In [ ]:
loudest_reading = mean_per_reading.argmax()
print(f"Reading {loudest_reading}, mean value {mean_per_reading[loudest_reading]:.4f}, label: {sonar_labels[loudest_reading]}")


---

## Resumen de la clase

- `dtype` controla tanto el uso de memoria como la corrección (truncamiento silencioso, desbordamiento silencioso) — siempre merece la pena conocerlo, no asumir sin más que el valor por defecto de NumPy es el adecuado para la tarea.
- Las funciones de creación (`zeros`/`ones`/`arange`/`linspace`/`meshgrid`/`identity`) cubren la mayoría de necesidades reales de construcción de arrays sin escribir elementos a mano.
- El indexado/slicing multidimensional extiende la propia sintaxis de slices de Python con una coma por eje.
- **El slicing básico devuelve una vista (comparte memoria); el indexado fancy y `.copy()` devuelven una copia genuina.** Esta única regla evita una clase de bug real, frecuente y silenciosa.
- Las máscaras booleanas y el indexado fancy seleccionan datos reales por condición o por una lista explícita de posiciones — la misma herramienta que sostiene cada filtro que ha usado este curso desde `NB02`.

## Para la próxima clase (NB04)

Pasamos de arrays sueltos a operaciones reales de matrices y vectores — broadcasting, `einsum`, y resolver sistemas reales de ecuaciones lineales con `numpy.linalg`, aplicado a un problema real de estática naval (tensiones de líneas de amarre).

## Tarea / Ideas de práctica

1. Carga `ship_fuel_efficiency.csv` (`NB07`) como un array de NumPy (no como un DataFrame) y usa máscaras booleanas para seleccionar solo las filas donde `ship_type == "Tanker Ship"`.
2. Demuestra tú mismo la distinción vista/copia: haz slicing de un subconjunto de 2 columnas de cualquier array de aquí, modifícalo, y comprueba con `np.shares_memory` si el original cambió.
3. Usa `np.meshgrid` para construir una rejilla 5x5 de coordenadas (x, y) que vaya de -2 a 2 en ambas direcciones, y calcula `z = x**2 + y**2` sobre toda la rejilla a la vez (sin bucle).
4. Compara la huella de memoria real (`.nbytes`) del dataset Sonar almacenado como `float64`, `float32` y `float16` — ¿en qué punto (si lo hay) empieza `np.allclose` a reportar diferencias numéricas reales?
5. Usando indexado fancy, construye una copia "barajada" del orden de filas del dataset Sonar (pista: `np.random.permutation`) y confirma con `np.shares_memory` que es una copia genuina, no una vista.

> ***Como siempre: un dataset real ya usado en otra parte de este curso es exactamente lo que hace que un ejercicio de mecánica de arrays sea concreto en vez de abstracto.***